<a href="https://colab.research.google.com/github/chitrita01/AI-Powered-Text-Categorization-and-sentiment-analysis/blob/main/categorization_%26_sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

#categorization and sentiment analysis



import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import re
import string
import nltk
nltk.download('stopwords',quiet = 'True')
nltk.download('punkt_tab')
nltk.download('wordnet')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
#!pip install -q textblob
from textblob import TextBlob,Word
import warnings
warnings.filterwarnings('ignore')
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
!pip install kneed
from kneed import KneeLocator

dataset = pd.read_csv("/content/sentimentdataset.csv")

dataset.head(10)

dataset.shape
dataset.info()

df = dataset.drop(['Timestamp','Month','Day','Hour','User','Platform','Hashtags','Retweets','Likes',],axis=1)
df.head()

df.shape

# Preprocessing the data


def preprocess_text(text):

    # converting to lowercase
    text = text.lower()

    # removing numbers and emojis
    text = re.sub(r'\b[0-9]+\b\s*|[^\w\s,]', '', text)

    # removing punctuations
    text = ''.join([char for char in text if char not in string.punctuation])

    # Correcting spellings if any using TextBlob
    blob = TextBlob(text)
    corr_text = blob.correct()
    new_text = str(corr_text)

    # Applying tokenization
    tokens = word_tokenize(new_text)

    # Stopwords removal
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]

    # Convert to lowercase
    tokens = [token.lower() for token in tokens]

    # Lemmatize the words
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in word_tokenize(new_text)]

    # Join tokens back into a single string
    processed_text = ' '.join(tokens)

    return processed_text

# Apply preprocessing to the 'Feedback' column
df['Processed_Text'] = df['Text'].apply(preprocess_text)

df['Processed_Text']

from sentence_transformers import SentenceTransformer


model = SentenceTransformer("all-MiniLM-L6-v2")

# Tokenize each text using the model's tokenizer
df["Token_Count"] = df["Processed_Text"].apply(lambda x: len(model.tokenizer.tokenize(x)))

"""EMBEDDING"""

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(df["Processed_Text"].tolist(), show_progress_bar=True)

embeddings = np.array(embeddings)

print(embeddings)



distortions = []
K = range(1, 51)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(embeddings)
    distortions.append(kmeans.inertia_)


plt.figure(figsize=(8, 6))
plt.plot(K, distortions, marker='o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Distortion (Inertia)')
plt.title('Elbow Method for Optimal k')
plt.grid(True)
plt.show()



kl = KneeLocator(K, distortions, curve='convex', direction='decreasing')
optimal_k = kl.elbow
print(f"\n Optimal number of clusters (elbow method): {optimal_k}")



kmeans_optimal = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df["Cluster"] = kmeans_optimal.fit_predict(embeddings)



print("\n Cluster Distribution:\n", df["Cluster"].value_counts())

"""MERGING"""

cluster_texts = df.groupby("Cluster")["Processed_Text"].apply(lambda x: " ".join(x)).to_dict()

# Print sample text for each cluster
for cluster, text in cluster_texts.items():
    print(f"\n Cluster {cluster} Sample Text:\n{text[:500]}...")

from sklearn.cluster import KMeans
from kneed import KneeLocator
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification


model_name = "ml6team/keyphrase-extraction-kbir-inspec"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)


def extract_keyphrases(text, max_length=512):
    tokens = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=max_length)
    with torch.no_grad():
        outputs = model(**tokens).logits
    predictions = torch.argmax(outputs, dim=-1).squeeze().tolist()
    input_ids = tokens["input_ids"].squeeze().tolist()

    keyphrases = [tokenizer.decode([input_ids[idx]]) for idx, pred in enumerate(predictions) if pred == 1]
    return list(set(keyphrases))

# Convert to DataFrame
cluster_df = pd.DataFrame(list(cluster_texts.items()), columns=["Cluster", "Merged_Text"])



# Apply KBIR model to extract keyphrases per cluster
print("Extracting keyphrases for each cluster...")
cluster_df["Keywords"] = cluster_df["Merged_Text"].apply(lambda text: extract_keyphrases(text[:1000]))



print("\n Sample Keyphrases per Cluster:")
for idx, row in cluster_df.iterrows():
    print(f"\nCluster {row['Cluster']} : {row['Keywords'][:10]}")



print("\n Top 5 Keyword Frequencies per Cluster:")
for cluster, keywords in zip(cluster_df["Cluster"], cluster_df["Keywords"]):
    if keywords:
        freq = pd.Series(keywords).value_counts().head(5).to_dict()
        print(f"\nCluster {cluster}: {freq}")
    else:
        print(f"\nCluster {cluster}: No keywords extracted.")

def name_cluster(keywords, top_n=3):

    if not keywords:
        return "No Keywords"
    # Count frequency and pick top N keywords
    freq = pd.Series(keywords).value_counts().head(top_n).index.tolist()
    return ", ".join(freq)

# Create a new column with Cluster Names
cluster_df["Cluster_Name"] = cluster_df["Keywords"].apply(lambda kw: name_cluster(kw, top_n=3))



print("\n Cluster Names Based on Top Keywords:")
for idx, row in cluster_df.iterrows():
    print(f"Cluster {row['Cluster']} -> {row['Cluster_Name']}")

def name_cluster_single(keywords):

    if not keywords:
        return "No Keywords"
    # Get the most frequent keyword
    most_common = pd.Series(keywords).value_counts().idxmax()
    return most_common
# Assign a single name per cluster using the most frequent keyword
cluster_df["Cluster_Name"] = cluster_df["Keywords"].apply(name_cluster_single)

# Display cluster names
print("\nCluster Names Based on Most Frequent Keyword:")
for idx, row in cluster_df.iterrows():
    print(f"Cluster {row['Cluster']} -> {row['Cluster_Name']}")

cluster_name_map = dict(zip(cluster_df["Cluster"], cluster_df["Cluster_Name"]))

df["Cluster_Name"] = df["Cluster"].map(cluster_name_map)

print(df[["Text", "Cluster", "Cluster_Name"]].head(10))



!pip install kneed



import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from kneed import KneeLocator
import joblib
import os

# Example: assuming embeddings is already defined
# embeddings = ...

# Step 1: Find distortions for different k
distortions = []
K = range(1, 51)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(embeddings)
    distortions.append(kmeans.inertia_)

# Plot Elbow Curve
plt.figure(figsize=(8, 6))
plt.plot(K, distortions, marker='o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Distortion (Inertia)')
plt.title('Elbow Method for Optimal k')
plt.grid(True)
plt.show()

# Step 2: Use KneeLocator to find optimal_k
kl = KneeLocator(K, distortions, curve='convex', direction='decreasing')
optimal_k = kl.elbow
print(f"\nOptimal number of clusters (elbow method): {optimal_k}")

# Step 3: Fit KMeans with optimal_k
kmeans_optimal = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
kmeans_optimal.fit(embeddings)

# Step 4: Save the model
os.makedirs('/content/saved_models', exist_ok=True)
save_path = '/content/saved_models/kmeans_model.pkl'
joblib.dump(kmeans_optimal, save_path)
print(f"Model saved at: {save_path}")

# Step 5: Download the model
from google.colab import files
files.download(save_path)

def save_model(self):
    joblib.dump(self.kmeans_model, "kmeans_model.pkl")
    self.model.save("embedding_model")  # Correct way to save SentenceTransformer
    joblib.dump(self.cluster_name_map, "cluster_names.pkl")
    self.df.to_csv("trained_dataset.csv", index=False)

"""final inference script"""



import pandas as pd
import numpy as np
import re
import string
import nltk
import torch
import joblib
import os
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from kneed import KneeLocator
from transformers import AutoTokenizer, AutoModelForTokenClassification

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)

# Temporary storage for 30 texts
temp_storage = []

# Preprocessing function
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'\b[0-9]+\b\s*|[^\w\s,]', '', text)
    text = ''.join([char for char in text if char not in string.punctuation])
    blob = TextBlob(text)
    corr_text = blob.correct()
    new_text = str(corr_text)
    tokens = word_tokenize(new_text)
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]
    tokens = [token.lower() for token in tokens]
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    processed_text = ' '.join(tokens)
    return processed_text

# Keyphrase extraction setup
model_name = "ml6team/keyphrase-extraction-kbir-inspec"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

def extract_keyphrases(text, max_length=512):
    tokens = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=max_length)
    with torch.no_grad():
        outputs = model(**tokens).logits
    predictions = torch.argmax(outputs, dim=-1).squeeze().tolist()
    input_ids = tokens["input_ids"].squeeze().tolist()
    keyphrases = [tokenizer.decode([input_ids[idx]]) for idx, pred in enumerate(predictions) if pred == 1]
    return list(set(keyphrases))

# Training Class
class TrainModel:
    def __init__(self, df):
        self.df = df
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.kmeans_model = None
        self.cluster_name_map = {}

    def train(self):
        print("\n[INFO] Preprocessing text...")
        self.df['Processed_Text'] = self.df['Text'].apply(preprocess_text)

        print("[INFO] Generating embeddings...")
        embeddings = self.model.encode(self.df['Processed_Text'].tolist(), show_progress_bar=True)

        distortions = []
        K = range(1, 51)
        for k in K:
            kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
            kmeans.fit(embeddings)
            distortions.append(kmeans.inertia_)

        kl = KneeLocator(K, distortions, curve='convex', direction='decreasing')
        optimal_k = kl.elbow
        print(f"[INFO] Optimal clusters found: {optimal_k}")

        self.kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
        self.df["Cluster"] = self.kmeans_model.fit_predict(embeddings)

        cluster_texts = self.df.groupby("Cluster")["Processed_Text"].apply(lambda x: " ".join(x)).to_dict()
        cluster_df = pd.DataFrame(list(cluster_texts.items()), columns=["Cluster", "Merged_Text"])

        print("[INFO] Extracting keyphrases for each cluster...")
        cluster_df["Keywords"] = cluster_df["Merged_Text"].apply(lambda text: extract_keyphrases(text[:1000]))

        def name_cluster_single(keywords):
            if not keywords:
                return "No Keywords"
            most_common = pd.Series(keywords).value_counts().idxmax()
            return most_common

        cluster_df["Cluster_Name"] = cluster_df["Keywords"].apply(name_cluster_single)
        print("\n[INFO] Cluster Names Based on Most Frequent Keyword:")
        for idx, row in cluster_df.iterrows():
            print(f"Cluster {row['Cluster']} -> {row['Cluster_Name']}")

        self.cluster_name_map = dict(zip(cluster_df["Cluster"], cluster_df["Cluster_Name"]))
        self.df["Cluster_Name"] = self.df["Cluster"].map(self.cluster_name_map)

        self.save_model()
        print("[INFO] Training complete. Model and mappings saved.")

    def save_model(self):
        joblib.dump(self.kmeans_model, "kmeans_model.pkl")
        self.model.save("embedding_model")
        joblib.dump(self.cluster_name_map, "cluster_names.pkl")
        self.df.to_csv("trained_dataset.csv", index=False)
        print("[INFO] Model saved successfully.")

# Inference Class
class InferenceModel:
    def __init__(self):
        self.model = SentenceTransformer("embedding_model")
        self.kmeans_model = joblib.load("kmeans_model.pkl")
        self.cluster_name_map = joblib.load("cluster_names.pkl")

    def get_sentiment(self, text):
        polarity = TextBlob(text).sentiment.polarity
        if polarity > 0:
            return "Positive "
        elif polarity < 0:
            return "Negative "
        else:
            return "Neutral "

    def infer_single_text(self, text):
        processed_text = preprocess_text(text)
        embedding = self.model.encode([processed_text], show_progress_bar=False)
        cluster = self.kmeans_model.predict(embedding)[0]
        cluster_name = self.cluster_name_map.get(cluster, "Unknown")
        sentiment = self.get_sentiment(text)

        print(f"\nInput Text: {text}")
        print(f"Predicted Cluster: {cluster} -> {cluster_name}")
        print(f"Sentiment: {sentiment}")

        temp_storage.append(text)
        if len(temp_storage) >= 30:
            print("[INFO] Temporary storage full. Adding to dataset and retraining...")
            append_and_train_new_data()

    def infer_batch(self, df):
        print("\n[INFO] Preprocessing text for batch inference...")
        df['Processed_Text'] = df['Text'].apply(preprocess_text)

        print("[INFO] Generating embeddings...")
        embeddings = self.model.encode(df['Processed_Text'].tolist(), show_progress_bar=False)

        print("[INFO] Predicting clusters...")
        clusters = self.kmeans_model.predict(embeddings)
        df["Cluster"] = clusters
        df["Cluster_Name"] = df["Cluster"].map(self.cluster_name_map)
        df["Sentiment"] = df["Text"].apply(self.get_sentiment)

        print(df[["Text", "Cluster", "Cluster_Name", "Sentiment"]])

# Function to append new data and retrain
def append_and_train_new_data(dataset_path="sentimentdataset.csv"):
    global temp_storage
    new_df = pd.DataFrame({'Text': temp_storage})

    # Load existing dataset if it exists
    if os.path.exists(dataset_path):
        existing_df = pd.read_csv(dataset_path)
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
    else:
        combined_df = new_df

    # Save new combined dataset
    combined_df.to_csv(dataset_path, index=False)
    print(f"[INFO] New dataset saved to {dataset_path} with {len(combined_df)} rows.")

    # Train model on updated dataset
    print("[INFO] Training model with updated dataset...")
    train_instance = TrainModel(combined_df)
    train_instance.train()

    # Clear temporary storage
    temp_storage = []

    # Perform inference on newly added data
    print("[INFO] Performing inference on newly added data...")
    inference_instance = InferenceModel()
    inference_instance.infer_batch(new_df)

# Main Function
def main():
    dataset_path = "sentimentdataset.csv"

    while True:
        print("\nOptions:\n1. Predict single/multiple texts\n0. Exit")
        choice = input("Enter your choice: ")

        if choice == "1":
            inference_instance = InferenceModel()
            num_texts = int(input("How many texts would you like to predict? (0 to cancel): "))
            if num_texts == 0:
                continue
            for i in range(num_texts):
                user_input = input(f"Enter text {i+1}: ")
                inference_instance.infer_single_text(user_input)

        elif choice == "0":
            print("Exiting...")
            break
        else:
            print("Invalid choice. Please enter 1 or 0.")

if __name__ == "__main__":
    main()

import pandas as pd
import numpy as np
import re
import string
import nltk
import torch
import joblib
import os
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from kneed import KneeLocator
from transformers import AutoTokenizer, AutoModelForTokenClassification

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)

# Temporary storage for 30 texts
temp_storage = []

# Preprocessing function
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'\b[0-9]+\b\s*|[^\w\s,]', '', text)
    text = ''.join([char for char in text if char not in string.punctuation])
    blob = TextBlob(text)
    corr_text = blob.correct()
    new_text = str(corr_text)
    tokens = word_tokenize(new_text)
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]
    tokens = [token.lower() for token in tokens]
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    processed_text = ' '.join(tokens)
    return processed_text

# Keyphrase extraction setup
model_name = "ml6team/keyphrase-extraction-kbir-inspec"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

def extract_keyphrases(text, max_length=512):
    tokens = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=max_length)
    with torch.no_grad():
        outputs = model(**tokens).logits
    predictions = torch.argmax(outputs, dim=-1).squeeze().tolist()
    input_ids = tokens["input_ids"].squeeze().tolist()
    keyphrases = [tokenizer.decode([input_ids[idx]]) for idx, pred in enumerate(predictions) if pred == 1]
    return list(set(keyphrases))

# Training Class
class TrainModel:
    def __init__(self, df):
        self.df = df
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.kmeans_model = None
        self.cluster_name_map = {}

    def train(self):
        print("\n[INFO] Preprocessing text...")
        self.df['Processed_Text'] = self.df['Text'].apply(preprocess_text)

        print("[INFO] Generating embeddings...")
        embeddings = self.model.encode(self.df['Processed_Text'].tolist(), show_progress_bar=True)

        distortions = []
        K = range(1, 51)
        for k in K:
            kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
            kmeans.fit(embeddings)
            distortions.append(kmeans.inertia_)

        kl = KneeLocator(K, distortions, curve='convex', direction='decreasing')
        optimal_k = kl.elbow
        print(f"[INFO] Optimal clusters found: {optimal_k}")

        self.kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
        self.df["Cluster"] = self.kmeans_model.fit_predict(embeddings)

        cluster_texts = self.df.groupby("Cluster")["Processed_Text"].apply(lambda x: " ".join(x)).to_dict()
        cluster_df = pd.DataFrame(list(cluster_texts.items()), columns=["Cluster", "Merged_Text"])

        print("[INFO] Extracting keyphrases for each cluster...")
        cluster_df["Keywords"] = cluster_df["Merged_Text"].apply(lambda text: extract_keyphrases(text[:1000]))

        def name_cluster_single(keywords):
            if not keywords:
                return "No Keywords"
            most_common = pd.Series(keywords).value_counts().idxmax()
            return most_common

        cluster_df["Cluster_Name"] = cluster_df["Keywords"].apply(name_cluster_single)
        print("\n[INFO] Cluster Names Based on Most Frequent Keyword:")
        for idx, row in cluster_df.iterrows():
            print(f"Cluster {row['Cluster']} -> {row['Cluster_Name']}")

        self.cluster_name_map = dict(zip(cluster_df["Cluster"], cluster_df["Cluster_Name"]))
        self.df["Cluster_Name"] = self.df["Cluster"].map(self.cluster_name_map)

        self.save_model()
        print("[INFO] Training complete. Model and mappings saved.")

    def save_model(self):
        joblib.dump(self.kmeans_model, "kmeans_model.pkl")
        self.model.save("embedding_model")
        joblib.dump(self.cluster_name_map, "cluster_names.pkl")
        self.df.to_csv("trained_dataset.csv", index=False)
        print("[INFO] Model saved successfully.")

# Inference Class
class InferenceModel:
    def __init__(self):
        self.model = SentenceTransformer("embedding_model")
        self.kmeans_model = joblib.load("kmeans_model.pkl")
        self.cluster_name_map = joblib.load("cluster_names.pkl")

    def get_sentiment(self, text):
        polarity = TextBlob(text).sentiment.polarity
        if polarity > 0:
            return "Positive "
        elif polarity < 0:
            return "Negative "
        else:
            return "Neutral "

    def infer_single_text(self, text):
        processed_text = preprocess_text(text)
        embedding = self.model.encode([processed_text], show_progress_bar=False)
        cluster = self.kmeans_model.predict(embedding)[0]
        cluster_name = self.cluster_name_map.get(cluster, "Unknown")
        sentiment = self.get_sentiment(text)

        print(f"\nInput Text: {text}")
        print(f"Predicted Cluster: {cluster} -> {cluster_name}")
        print(f"Sentiment: {sentiment}")

        temp_storage.append(text)
        if len(temp_storage) >= 30:
            print("[INFO] Temporary storage full. Adding to dataset and retraining...")
            append_and_train_new_data()

    def infer_batch(self, df):
        print("\n[INFO] Preprocessing text for batch inference...")
        df['Processed_Text'] = df['Text'].apply(preprocess_text)

        print("[INFO] Generating embeddings...")
        embeddings = self.model.encode(df['Processed_Text'].tolist(), show_progress_bar=False)

        print("[INFO] Predicting clusters...")
        clusters = self.kmeans_model.predict(embeddings)
        df["Cluster"] = clusters
        df["Cluster_Name"] = df["Cluster"].map(self.cluster_name_map)
        df["Sentiment"] = df["Text"].apply(self.get_sentiment)

        print(df[["Text", "Cluster", "Cluster_Name", "Sentiment"]])

# Function to append new data and retrain
def append_and_train_new_data(dataset_path="sentimentdataset.csv"):
    global temp_storage
    new_df = pd.DataFrame({'Text': temp_storage})

    # Load existing dataset if it exists
    if os.path.exists(dataset_path):
        existing_df = pd.read_csv(dataset_path)
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
    else:
        combined_df = new_df

    # Save new combined dataset
    combined_df.to_csv(dataset_path, index=False)
    print(f"[INFO] New dataset saved to {dataset_path} with {len(combined_df)} rows.")

    # Train model on updated dataset
    print("[INFO] Training model with updated dataset...")
    train_instance = TrainModel(combined_df)
    train_instance.train()

    # Clear temporary storage
    temp_storage = []

    # Perform inference on newly added data
    print("[INFO] Performing inference on newly added data...")
    inference_instance = InferenceModel()
    inference_instance.infer_batch(new_df)

# Main Function
def main():
    dataset_path = "sentimentdataset.csv"

    while True:
        print("\nOptions:\n1. Predict single/multiple texts\n0. Exit")
        choice = input("Enter your choice: ")

        if choice == "1":
            inference_instance = InferenceModel()
            num_texts = int(input("How many texts would you like to predict? (0 to cancel): "))
            if num_texts == 0:
                continue
            for i in range(num_texts):
                user_input = input(f"Enter text {i+1}: ")
                inference_instance.infer_single_text(user_input)

        elif choice == "0":
            print("Exiting...")
            break
        else:
            print("Invalid choice. Please enter 1 or 0.")

if __name__ == "__main__":
    main()

